---
title: xarray untuk Open Data Cube
short_title: xarray untuk ODC
subject: Panduan Pemula
subtitle: Bekerja dengan xarray.Dataset yang dihasilkan oleh dc.load.
description: Bekerja dengan xarray.Dataset yang dihasilkan oleh dc.load.
keywords:
  - open-data-cube
  - odc
  - xarray
  - beginner-guide
---

Notebook ini memperkenalkan `xarray.Dataset` dan `xarray.DataArray`, dua objek yang menampung data dari `dc.load`.
Setelah memahami label dan susunan datanya, subset dapat dipilih dan variabel baru dapat dihitung dengan lebih tepat.[^edits]

[^edits]: Notebook tutorial diperbarui secara otomatis; perubahan pada notebook tutorial dapat tertimpa pada pembaruan berikutnya.
    Simpan salinan kerja dalam berkas terpisah agar perubahan tidak hilang.

## A. Tujuan

- Memuat `Dataset` contoh
- Memahami dimensi, koordinat, dan variabel datanya
- Memilih subset berdasarkan label dan indeks
- Menghitung `DataArray` baru dengan operasi aritmetika

## B. Dataset dan DataArray xarray

Satu observasi satelit merekam kisi nilai piksel untuk satu band pada satu waktu.
Kueri data pengamatan Bumi biasanya meminta beberapa band dan tanggal pada area yang sama, sehingga hasilnya terdiri atas sekumpulan kisi yang harus tetap sejajar.

Array numerik biasa menyimpan nilai, tetapi tidak menjelaskan arti setiap sumbu dan posisi di dalamnya.
Tanggal, posisi piksel, nama band, dan informasi proyeksi harus dicatat secara terpisah.
xarray menyimpan keterangan tersebut bersama nilainya agar data lebih mudah diperiksa dan tidak keliru saat digabungkan.

Setiap sumbu bernama disebut **dimensi**.
Data dalam notebook ini memakai dimensi `time`, `y`, dan `x`: `time` menunjukkan waktu observasi, sedangkan y dan x menunjukkan letak piksel pada kisi spasial.
**Koordinat** menyediakan label di sepanjang dimensi, misalnya tanggal pada `time` serta posisi hasil proyeksi pada `y` dan `x`.

xarray menyusun data berlabel tersebut dalam dua objek utama:

- **DataArray** menampung satu variabel bernama beserta dimensi dan koordinatnya.
  Dalam notebook ini, band seperti `red` merupakan DataArray dengan dimensi `time`, `y`, dan `x`.
  Karena labelnya tetap melekat, nilai dari tanggal dan posisi piksel yang berbeda tidak tertukar saat dipilih atau dihitung.

- **Dataset** menghimpun beberapa DataArray yang saling berkaitan.
  Setiap band yang diminta (`red`, `green`, `blue`, dan `nir`) menjadi variabel data dalam Dataset hasil `dc.load`.
  Semua variabel tersebut memakai kisi `time`, `y`, dan `x` yang sama, sehingga nilai pada koordinat yang sama merujuk pada tanggal dan lokasi yang sama.
  Dataset juga dapat menyimpan atribut yang menjelaskan seluruh kumpulan data, termasuk proyeksi petanya.

![Satu DataArray digambarkan sebagai satu kisi berlabel pada y/lintang dan x/bujur, berdampingan dengan satu Dataset yang digambarkan sebagai matriks empat band (red, green, blue, nir) pada empat tanggal, dengan semua band menggunakan kisi y dan x yang sama.](../../../assets/xarray-dataarray-vs-dataset.svg)

`dc.load` menghasilkan sebuah Dataset xarray.
Bagian selanjutnya memuat contoh berukuran kecil, lalu menguraikan informasi yang tampil di dalamnya.
Dokumentasi xarray memuat definisi formal kedua objek tersebut: [struktur data xarray](https://docs.xarray.dev/en/stable/user-guide/data-structures.html).

## C. Memuat Dataset

Kode berikut memuat data GeoMAD tahunan untuk empat band pada area kecil.
Hasilnya disimpan sebagai `ds`, singkatan yang lazim digunakan untuk dataset.

In [ ]:
from datacube import Datacube

dc = Datacube(app="xarray_for_odc")

query = {
    "product": "s2_geomad_annual",
    "x": (98.80, 98.90),
    "y": (2.65, 2.55),
    "time": ("2024", "2025"),
    "measurements": ["red", "green", "blue", "nir"],
    "output_crs": "EPSG:32647",
    "resolution": (-30, 30),
}

ds = dc.load(**query)
ds

## D. Membaca Struktur Dataset

Tampilan Dataset memisahkan keterangannya menjadi dimensi, variabel data, koordinat, dan atribut.
Periksa bagian-bagian ini sebelum memulai analisis agar isi data sesuai dengan kueri yang diminta.

![Struktur xarray.Dataset yang dihasilkan oleh dc.load, dengan empat bagian berupa dimensi, variabel data, koordinat, dan atribut.](../../../assets/xarray-dataset-structure.svg)

1. **Dimensi dan ukuran** menjelaskan sumbu data beserta panjangnya.

In [ ]:
ds.sizes

`time`, `y`, dan `x` adalah nama dimensinya.
Pada hasil ini, ukurannya menunjukkan dua irisan waktu pada kisi berukuran 369 × 372 piksel.
Bersama resolusi 30 m yang diminta, informasi tersebut memberi gambaran awal tentang cakupan area dan banyaknya data yang akan diolah.

2. **Variabel data** berisi nilai pengukuran dalam Dataset.

In [ ]:
ds.data_vars

Empat variabel datanya adalah band reflektansi permukaan merah, hijau, biru, dan inframerah dekat yang diminta dalam kueri.
Untuk setiap variabel, tampilan ini mencantumkan dimensi, bentuk, tipe data, perkiraan ukuran, dan beberapa contoh nilai.
Rincian tersebut membantu mengenali cara suatu pengukuran disimpan, misalnya apakah suhu permukaan masih berupa angka digital mentah atau sudah dikonversi ke derajat.

Pilih satu variabel dari `data_vars` untuk melihat DataArray beserta dimensi, koordinat, nilai, dan atributnya:

In [ ]:
ds.data_vars["red"]

3. **Koordinat** menyediakan label untuk menemukan data di sepanjang setiap dimensi.

In [ ]:
ds.coords

Koordinat `time` memuat tanggal, sedangkan `y` dan `x` memuat posisi piksel hasil proyeksi.
Label inilah yang dipakai saat memilih data dan menyelaraskan nilai dari tanggal serta lokasi yang sama dalam suatu perhitungan.

4. **Atribut** menyimpan metadata yang berlaku untuk seluruh Dataset.

In [ ]:
ds.attrs

CRS yang diminta adalah EPSG:32647 (WGS 84 / UTM zone 47N).
CRS terproyeksi ini memakai satuan meter, sehingga selisih nilai pada koordinat `y` dan `x` dapat dibaca sebagai jarak tanpa melakukan reproyeksi terlebih dahulu.

## E. Memilih Subset

Pemilihan subset mengambil bagian Dataset atau DataArray yang diperlukan untuk perhitungan berikutnya tanpa mengubah `ds` asli.
Gunakan `.sel` jika posisi diketahui dari label koordinatnya, atau `.isel` jika posisi diketahui dari indeks integernya.

Keduanya merupakan metode, yaitu fungsi yang melekat pada objek dan dipanggil dengan notasi titik.
Sebagai contoh, `ds.sel(time="2024")` memakai label waktu, sedangkan `ds.isel(time=0)` memakai posisi.
Di dalam setiap pemanggilan, `time=value` menyatakan dimensi serta label atau indeks yang hendak dipertahankan.

### 1. Pemilihan berdasarkan label

`.sel` mencari data berdasarkan label koordinat.
xarray mengenali label waktu, sehingga dimensi `time` dapat dipilih dengan tahun, tanggal lengkap, daftar tanggal, atau rentang tanggal.

In [ ]:
ds.sel(time="2024")

Label tahun tersebut memilih koordinat waktu yang berada pada 2024.
Hasilnya berupa Dataset baru dengan irisan waktu tersebut, sementara kisi y dan x tetap sama.

Pola pemilihan berdasarkan label lainnya memakai metode yang sama:

- daftar label: `ds.sel(time=["2024-01-01", "2025-01-01"])`
- rentang: `ds.sel(time=slice("2024", "2025"))`
- mask boolean: `ds.sel(time=ds.time.dt.year == 2024)`

### 2. Pemilihan berdasarkan indeks

`.isel` mencari data berdasarkan posisi integer, mengikuti indeks berbasis nol seperti NumPy.

In [ ]:
ds.isel(time=0)

Indeks `0` memilih posisi pertama pada dimensi `time` tanpa bergantung pada label tanggalnya.
Daftar indeks (`ds.isel(time=[0, 1])`) atau irisan (`ds.isel(time=slice(0, 2))`) dapat dipakai untuk memilih beberapa posisi.
Cara ini berguna ketika posisinya diketahui, tetapi label koordinatnya tidak.

## F. Aritmetika Antarvariabel

Setiap variabel data dalam Dataset merupakan DataArray dan dapat diakses melalui namanya, seperti `ds.red` atau `ds.nir`.
Operator `+`, `-`, `*`, dan `/` menghitung nilai per elemen.
Sebelum menghitung, xarray menyelaraskan DataArray berdasarkan nama dimensi dan label koordinatnya agar nilai dari tanggal atau lokasi yang berbeda tidak tergabung secara keliru.

**Simple Ratio (SR)** membandingkan reflektansi inframerah dekat dengan reflektansi merah:

$$\text{SR} = \frac{\text{NIR}}{\text{Red}}$$

Vegetasi sehat memantulkan inframerah dekat dengan kuat dan menyerap cahaya merah.
Karena itu, nilai SR biasanya tinggi (> 1) pada hutan dan tanaman, mendekati 1 pada tanah terbuka, serta rendah pada perairan.

In [ ]:
sr = ds.nir / ds.red
sr

Pada setiap koordinat waktu dan piksel, perhitungan ini membagi nilai inframerah dekat dengan nilai merah yang bersesuaian.
Hasilnya, `sr`, merupakan DataArray baru dengan dimensi dan koordinat `time`, `y`, dan `x` yang sama.

## G. Langkah Berikutnya

Notebook 05 menerapkan struktur tersebut untuk menggambar komposit warna alami dan citra band tunggal: [`05_visualisasi_data_xarray.ipynb`](./05_visualisasi_data_xarray.ipynb).